In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import math
from functools import reduce
import scienceplots

In [ ]:
# -------------------------
# Basic definitions
# -------------------------
ket_0 = np.array([1, 0], dtype=complex)
I = np.eye(2, dtype=complex)

Pauli_X = np.array([[0, 1], [1, 0]], dtype=complex)
Pauli_Z = np.array([[1, 0], [0, -1]], dtype=complex)

def kron_power(v, N):
    return reduce(np.kron, [v] * N)

def local_operator(op, i, N):
    ops = [I] * N
    ops[i] = op
    return reduce(np.kron, ops)

def stabilizer_operator(pauli, sites, N):
    """
    Product of the same Pauli operator over given qubit sites.
    Example: X_i X_j X_k X_l
    """
    O = np.eye(2**N, dtype=complex)

    for site in sites:
        O = local_operator(pauli, site, N) @ O

    return O

def plus_projector(S):
    """
    Projector onto the +1 eigenspace of stabilizer S.
    """
    d = S.shape[0]
    return (np.eye(d, dtype=complex) + S) / 2


# -------------------------
# Full stabilizer ground state
# -------------------------
def toric_code_ground_state(N, x_plaquettes, z_stars):
    """
    Construct a toric-code ground state using the mathematically correct
    stabilizer projectors:

        P_S = (I + S) / 2

    x_plaquettes: list of tuples, e.g. [(0,1,2,3), ...]
    z_stars: list of tuples, e.g. [(0,3,4,7), ...]
    """

    psi = kron_power(ket_0, N)

    # Project onto +1 eigenspace of X plaquettes
    for sites in x_plaquettes:
        A = stabilizer_operator(Pauli_X, sites, N)
        psi = plus_projector(A) @ psi

    # Project onto +1 eigenspace of Z stars
    for sites in z_stars:
        B = stabilizer_operator(Pauli_Z, sites, N)
        psi = plus_projector(B) @ psi

    # Normalize final state
    norm = np.linalg.norm(psi)

    if norm < 1e-12:
        raise ValueError("Projection produced the zero vector. Check stabilizers or initial state.")

    psi = psi / norm

    return psi


def toric_code_rho_0(N, x_plaquettes, z_stars):
    psi = toric_code_ground_state(N, x_plaquettes, z_stars)
    rho_0 = np.outer(psi, psi.conj())
    return rho_0

In [ ]:
N = 12

x_plaquettes = [
    (0, 1, 2, 3),
    (3, 4, 5, 6),
    (6, 7, 8, 9),
    (9, 10, 11, 1),
]

z_stars = [
    (1,3,4),
    (2,3,5),
    (4,6,7),
    (5,6,8),
    (7,9,10),
    (8,9,11),
    (10,0,1),
    (11,0,2)
]

In [ ]:
psi_0 = toric_code_ground_state(N, x_plaquettes, z_stars)
rho_0 = toric_code_rho_0(N, x_plaquettes, z_stars)

print("State norm:", np.linalg.norm(psi_0))
print("Trace rho_0:", np.trace(rho_0))
print("rho_0 shape:", rho_0.shape)

print(rho_0 )